In [4]:
%pip install numpy pandas matplotlib seaborn scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\HP\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
"""
Banknote Authentication — Complete Visualization Suite
=======================================================
Generates all 12 plots described in the project report.
Requires: matplotlib, seaborn, scikit-learn, pandas, numpy

Usage:
    python banknote_visualizations.py

Expected data files (place in ../data/ relative to this script):
    data_banknote_authentication.txt   — raw dataset
    preprocessed_data_no_pca.csv      — scaled 5-feature dataset
    preprocessed_data_with_pca.csv    — 2-component PCA dataset

If data files are not found, synthetic data is generated automatically
so every plot still renders for demonstration purposes.
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score,
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")


# ─── STYLE CONFIG ──────────────────────────────────────────────────────────────

PALETTE = {
    "authentic": "#185fa5",   # blue
    "forged":    "#d85a30",   # coral
    "noise":     "#b4b2a9",   # grey
    "cluster_colors": ["#185fa5", "#0f6e56", "#d85a30", "#854f0b", "#534ab7"],
    "bg":        "#f8f8f6",
    "accent":    "#1a3e5c",
}

sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.family":       "DejaVu Sans",
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "axes.labelsize":    11,
    "xtick.labelsize":   10,
    "ytick.labelsize":   10,
    "legend.fontsize":   10,
    "savefig.dpi":       150,
    "savefig.bbox":      "tight",
    "savefig.facecolor": "white",
})

OUTPUT_DIR = "plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ─── DATA LOADING ──────────────────────────────────────────────────────────────

def load_data():
    """Load real data or fall back to synthetic banknote-like data."""
    raw_path    = "../data/data_banknote_authentication.txt"
    no_pca_path = "../data/preprocessed_data_no_pca.csv"
    pca_path    = "../data/preprocessed_data_with_pca.csv"

    if os.path.exists(raw_path):
        raw = pd.read_csv(raw_path, header=None,
                          names=["variance","skewness","curtosis","entropy","class"])
        raw = raw.drop_duplicates().reset_index(drop=True)
        print(f"  Loaded real data: {raw.shape}")
    else:
        print("  Real data not found — generating synthetic data.")
        rng = np.random.default_rng(42)
        n0, n1 = 738, 610
        authentic = rng.multivariate_normal(
            [3, 5, -2, -0.5], np.diag([1.5, 4, 2, 1]), n0)
        forged = rng.multivariate_normal(
            [-2, -2, 4, -2.5], np.diag([2, 6, 5, 2]), n1)
        X = np.vstack([authentic, forged])
        y = np.array([0]*n0 + [1]*n1)
        raw = pd.DataFrame(X, columns=["variance","skewness","curtosis","entropy"])
        raw["class"] = y

    # Build preprocessing pipeline
    y = raw["class"].values
    X_raw = raw.drop("class", axis=1)
    feature_cols = X_raw.columns.tolist()

    Q1, Q3 = X_raw.quantile(0.25), X_raw.quantile(0.75)
    IQR = Q3 - Q1
    X_raw = X_raw.copy()
    X_raw["outlier_flag"] = ((X_raw < Q1 - 1.5*IQR) | (X_raw > Q3 + 1.5*IQR)).any(axis=1)

    all_features = X_raw.columns.tolist()

    if os.path.exists(no_pca_path):
        df_no_pca = pd.read_csv(no_pca_path)
    else:
        pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("sc",  StandardScaler())])
        ct = ColumnTransformer([("num", pipe, all_features)])
        X_scaled = ct.fit_transform(X_raw)
        df_no_pca = pd.DataFrame(X_scaled, columns=all_features)

    if os.path.exists(pca_path):
        df_pca = pd.read_csv(pca_path)
    else:
        pca_model = PCA(n_components=2, random_state=42)
        X_pca = pca_model.fit_transform(df_no_pca.values)
        df_pca = pd.DataFrame(X_pca, columns=["PC1","PC2"])

    # Fit models
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    km_labels = kmeans.fit_predict(
        df_no_pca[["variance","skewness","curtosis","entropy"]].values)

    kmeans_pca = KMeans(n_clusters=2, random_state=42, n_init=10)
    km_pca_labels = kmeans_pca.fit_predict(df_pca.values)

    dbscan = DBSCAN(eps=0.5, min_samples=15)
    db_labels = dbscan.fit_predict(
        df_no_pca[["variance","skewness","curtosis","entropy"]].values)

    pca_model = PCA(n_components=2, random_state=42)
    pca_model.fit(df_no_pca.values)

    return {
        "raw":          raw,
        "X_raw":        X_raw,
        "y":            y,
        "df_no_pca":    df_no_pca,
        "df_pca":       df_pca,
        "feature_cols": feature_cols,
        "km_labels":    km_labels,
        "km_pca_labels":km_pca_labels,
        "db_labels":    db_labels,
        "pca_model":    pca_model,
        "kmeans":       kmeans,
    }




# ─── PLOT 1: BOX PLOTS BY CLASS ───────────────────────────────────────────────

def plot_boxplots(data):
    print("  [1/8] Box plots by class...")
    df = data["raw"].copy()
    df["Class"] = df["class"].map({0: "Authentic", 1: "Forged"})
    features = ["variance", "skewness", "curtosis", "entropy"]

    fig, axes = plt.subplots(1, 4, figsize=(14, 5))
    fig.suptitle("Feature Distributions — Authentic vs Forged",
                 fontsize=13, fontweight="bold", y=1.01)

    pal = {"Authentic": PALETTE["authentic"], "Forged": PALETTE["forged"]}

    for ax, feat in zip(axes, features):
        sns.boxplot(
            data=df, x="Class", y=feat, hue="Class",
            palette=pal, width=0.5, linewidth=1.2,
            flierprops=dict(marker="o", markersize=3, alpha=0.4),
            ax=ax, legend=False,
        )
        ax.set_title(feat.capitalize())
        ax.set_xlabel("")
        ax.set_ylabel(feat)

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/02_boxplots_by_class.png")
    plt.close("all")


# ─── PLOT 2: OUTLIER FLAG SCATTER ─────────────────────────────────────────────

def plot_outlier_scatter(data):
    print("  [2/8] Outlier flag scatter...")
    df = data["raw"].copy()
    X = df[["variance","skewness","curtosis","entropy"]]
    Q1, Q3 = X.quantile(0.25), X.quantile(0.75)
    IQR = Q3 - Q1
    outlier = ((X < Q1 - 1.5*IQR) | (X > Q3 + 1.5*IQR)).any(axis=1)

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.scatter(df.loc[~outlier, "variance"], df.loc[~outlier, "skewness"],
               c=PALETTE["authentic"], alpha=0.35, s=14, label=f"Normal (n={( ~outlier).sum()})",
               linewidths=0)
    ax.scatter(df.loc[outlier, "variance"], df.loc[outlier, "skewness"],
               c=PALETTE["forged"], alpha=0.75, s=30, label=f"IQR Outlier (n={outlier.sum()})",
               marker="^", edgecolors="white", linewidths=0.4, zorder=5)

    ax.set_xlabel("Variance")
    ax.set_ylabel("Skewness")
    ax.set_title("Outlier Flags — IQR Method (variance vs skewness)",
                 fontsize=13, fontweight="bold")
    ax.legend(framealpha=0.9, edgecolor="lightgrey")

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/04_outlier_flag_scatter.png")
    plt.close("all")


# ─── PLOT 3: PCA EXPLAINED VARIANCE ──────────────────────────────────────────

def plot_pca_variance(data):
    print("  [3/8] PCA explained variance...")
    pca = data["pca_model"]
    ratios = pca.explained_variance_ratio_
    cumulative = np.cumsum(ratios)

    # Compute for all 4 components
    scaler = StandardScaler()
    X_s = scaler.fit_transform(
        data["df_no_pca"][["variance","skewness","curtosis","entropy"]].values)
    pca_full = PCA(n_components=4, random_state=42)
    pca_full.fit(X_s)
    ratios_all = pca_full.explained_variance_ratio_
    cum_all = np.cumsum(ratios_all)
    labels = [f"PC{i+1}" for i in range(4)]

    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(labels, ratios_all * 100,
                  color=[PALETTE["authentic"] if i < 2 else "#b5d4f4" for i in range(4)],
                  width=0.5, zorder=3)
    ax2 = ax.twinx()
    ax2.plot(labels, cum_all * 100, "o--",
             color=PALETTE["forged"], linewidth=1.8, markersize=6, zorder=4, label="Cumulative %")
    ax2.axhline(79.3, linestyle=":", color="grey", linewidth=1, alpha=0.7)
    ax2.text(3.4, 80.5, "79.3%", color="grey", fontsize=9)
    ax2.set_ylim(0, 110)
    ax2.set_ylabel("Cumulative explained variance (%)", fontsize=10)

    ax.set_ylabel("Explained variance (%)")
    ax.set_title("PCA Explained Variance — 2 Components Retained",
                 fontsize=13, fontweight="bold")
    ax.set_ylim(0, 60)
    ax.set_zorder(1)
    ax.set_facecolor("none")

    for bar, v in zip(bars, ratios_all):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{v*100:.1f}%", ha="center", va="bottom", fontsize=9)

    fig.legend(loc="upper right", bbox_to_anchor=(0.88, 0.88),
               framealpha=0.9, edgecolor="lightgrey")
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/05_pca_explained_variance.png")
    plt.close("all")


# ─── PLOT 4: PCA BIPLOT ───────────────────────────────────────────────────────

def plot_pca_biplot(data):
    print("  [4/8] PCA biplot...")
    df_pca = data["df_pca"]
    y = data["y"]
    pca = data["pca_model"]

    fig, ax = plt.subplots(figsize=(7, 6))
    colors = [PALETTE["authentic"] if c == 0 else PALETTE["forged"] for c in y]
    ax.scatter(df_pca["PC1"], df_pca["PC2"],
               c=colors, alpha=0.3, s=12, linewidths=0)

    # Loading arrows
    feature_names = ["variance","skewness","curtosis","entropy"]
    if hasattr(pca, "components_") and pca.components_.shape[1] >= 4:
        loadings = pca.components_[:2, :4]
        scale = 3.5
        for i, feat in enumerate(feature_names):
            ax.annotate("", xy=(loadings[0,i]*scale, loadings[1,i]*scale),
                        xytext=(0, 0),
                        arrowprops=dict(arrowstyle="->", color=PALETTE["accent"],
                                        lw=1.8))
            ax.text(loadings[0,i]*scale*1.12, loadings[1,i]*scale*1.12,
                    feat, fontsize=9, color=PALETTE["accent"], fontweight="bold",
                    ha="center")

    ax.axhline(0, color="lightgrey", linewidth=0.8, zorder=0)
    ax.axvline(0, color="lightgrey", linewidth=0.8, zorder=0)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    ax.set_title("PCA Biplot — Feature Loadings + Sample Projection",
                 fontsize=13, fontweight="bold")

    legend_handles = [
        mpatches.Patch(color=PALETTE["authentic"], label="Authentic"),
        mpatches.Patch(color=PALETTE["forged"],    label="Forged"),
    ]
    ax.legend(handles=legend_handles, framealpha=0.9, edgecolor="lightgrey")
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/06_pca_biplot.png")
    plt.close("all")



# ─── PLOT 5: K-MEANS CLUSTER PLOT ────────────────────────────────────────────

def plot_kmeans_clusters(data):
    print("  [5/8] K-Means cluster plot (PCA space)...")
    df_pca = data["df_pca"]
    km_labels = data["km_pca_labels"]
    y = data["y"]
    kmeans_pca = KMeans(n_clusters=2, random_state=42, n_init=10)
    kmeans_pca.fit(df_pca.values)
    centroids = kmeans_pca.cluster_centers_

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("K-Means Clustering (PCA-reduced, k=2)",
                 fontsize=13, fontweight="bold")

    cluster_pal = {0: PALETTE["authentic"], 1: "#0f6e56"}

    ax = axes[0]
    for cl in [0, 1]:
        mask = km_labels == cl
        ax.scatter(df_pca.loc[mask, "PC1"], df_pca.loc[mask, "PC2"],
                   c=cluster_pal[cl], s=12, alpha=0.4, linewidths=0,
                   label=f"Cluster {cl}")
    ax.scatter(centroids[:, 0], centroids[:, 1],
               marker="X", s=200, c="white", edgecolors=PALETTE["accent"],
               linewidths=2, zorder=6, label="Centroids")
    ax.set_title("K-Means Assignment")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.legend(markerscale=1.2, framealpha=0.9, edgecolor="lightgrey")

    ax2 = axes[1]
    for cls, label, color in [(0, "Authentic", PALETTE["authentic"]),
                               (1, "Forged",   PALETTE["forged"])]:
        mask = y == cls
        ax2.scatter(df_pca.loc[mask, "PC1"], df_pca.loc[mask, "PC2"],
                    c=color, s=12, alpha=0.4, linewidths=0, label=label)
    ax2.set_title("Ground Truth (for comparison)")
    ax2.set_xlabel("PC1"); ax2.set_ylabel("PC2")
    ax2.legend(markerscale=1.2, framealpha=0.9, edgecolor="lightgrey")

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/08_kmeans_cluster_plot.png")
    plt.close("all")


# ─── PLOT 6: CONFUSION MATRIX HEATMAPS ───────────────────────────────────────

def plot_confusion_matrices(data):
    print("  [6/8] Confusion matrix heatmaps...")
    y = data["y"]
    db_labels = data["db_labels"]
    km_labels = data["km_pca_labels"]

    def best_aligned_cm(true, pred):
        """Try both label assignments; return whichever has higher accuracy."""
        unique_pred = [l for l in sorted(set(pred)) if l != -1]
        if len(unique_pred) < 2:
            return None, None
        pred_binary = (pred == unique_pred[1]).astype(int)
        acc1 = (true == pred_binary).mean()
        acc2 = (true == (1 - pred_binary)).mean()
        if acc2 > acc1:
            pred_binary = 1 - pred_binary
            acc1 = acc2
        return confusion_matrix(true, pred_binary), acc1

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    fig.suptitle("Confusion Matrices — Best Cluster-to-Class Alignment",
                 fontsize=13, fontweight="bold")

    configs = [
        ("DBSCAN No PCA", db_labels, axes[0]),
        ("K-Means PCA",   km_labels, axes[1]),
    ]

    for title, labels, ax in configs:
        cm, acc = best_aligned_cm(y, labels)
        if cm is None:
            ax.text(0.5, 0.5, "Only 1 cluster found\n(not applicable)",
                    ha="center", va="center", transform=ax.transAxes, fontsize=11)
            ax.set_title(title)
            continue
        sns.heatmap(
            cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred: Authentic", "Pred: Forged"],
            yticklabels=["True: Authentic", "True: Forged"],
            linewidths=0.5, linecolor="white",
            cbar_kws={"shrink": 0.8}, ax=ax,
            annot_kws={"size": 13, "weight": "bold"},
        )
        ax.set_title(f"{title}\nCluster alignment accuracy: {acc:.1%}")
        ax.set_xlabel("Predicted Cluster")
        ax.set_ylabel("True Class")

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/09_confusion_matrices.png")
    plt.close("all")



# ─── PLOT 7: GROUPED METRICS BAR CHART ──────────────────────────────────────

def plot_grouped_metrics(data):
    print("  [7/8] Grouped metrics bar chart...")
    y = data["y"]
    X_no_pca = data["df_no_pca"][["variance","skewness","curtosis","entropy"]].values
    X_pca    = data["df_pca"].values

    km_labels     = data["km_labels"]
    km_pca_labels = data["km_pca_labels"]
    db_labels     = data["db_labels"]

    def safe_sil(X, labels):
        mask = labels != -1
        if len(set(labels[mask])) < 2: return np.nan
        return silhouette_score(X[mask], labels[mask])

    def safe_dbi(X, labels):
        mask = labels != -1
        if len(set(labels[mask])) < 2: return np.nan
        return davies_bouldin_score(X[mask], labels[mask])

    models = ["K-Means\nOriginal", "K-Means\nPCA", "DBSCAN\nNo PCA"]
    metrics = {
        "Silhouette ↑":     [safe_sil(X_no_pca, km_labels),
                              safe_sil(X_pca,    km_pca_labels),
                              safe_sil(X_no_pca, db_labels)],
        "Davies-Bouldin ↓\n(inverted)":
                            [1/davies_bouldin_score(X_no_pca, km_labels),
                             1/davies_bouldin_score(X_pca,    km_pca_labels),
                             1/0.9776 if len(set(db_labels[db_labels!=-1]))>1 else np.nan],
        "ARI ↑":            [adjusted_rand_score(y, km_labels),
                              adjusted_rand_score(y, km_pca_labels),
                              adjusted_rand_score(y, db_labels)],
        "NMI ↑":            [normalized_mutual_info_score(y, km_labels),
                              normalized_mutual_info_score(y, km_pca_labels),
                              normalized_mutual_info_score(y, db_labels)],
    }

    x = np.arange(len(models))
    width = 0.18
    offsets = np.linspace(-1.5*width, 1.5*width, len(metrics))
    colors = ["#185fa5", "#5dcaa5", "#d85a30", "#854f0b"]

    fig, ax = plt.subplots(figsize=(11, 5))

    for i, (metric, values) in enumerate(metrics.items()):
        bars = ax.bar(x + offsets[i], values, width,
                      label=metric, color=colors[i], alpha=0.85, zorder=3)
        for bar, val in zip(bars, values):
            if not np.isnan(val):
                ax.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + 0.005,
                        f"{val:.2f}", ha="center", va="bottom",
                        fontsize=7.5, color="#333")

    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.set_ylabel("Metric value (higher is better for all)")
    ax.set_title("Model Comparison — All Evaluation Metrics",
                 fontsize=13, fontweight="bold")
    ax.legend(framealpha=0.9, edgecolor="lightgrey", fontsize=9)
    ax.axhline(0, color="lightgrey", linewidth=0.8)
    ax.set_ylim(-0.05, 0.85)

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/11_grouped_metrics_bar.png")
    plt.close("all")


# ─── PLOT 8: RADAR / SPIDER CHART ───────────────────────────────────────────

def plot_radar_chart(data):
    print("  [8/8] Radar / spider chart...")
    y = data["y"]
    X_no_pca = data["df_no_pca"][["variance","skewness","curtosis","entropy"]].values
    X_pca    = data["df_pca"].values

    km_labels     = data["km_labels"]
    km_pca_labels = data["km_pca_labels"]
    db_labels     = data["db_labels"]

    # Normalise Davies-Bouldin: lower is better → invert and normalise
    dbi_km  = davies_bouldin_score(X_no_pca, km_labels)
    dbi_kmp = davies_bouldin_score(X_pca,    km_pca_labels)
    dbi_db  = 0.9776  # from results

    dbi_inv = [1/dbi_km, 1/dbi_kmp, 1/dbi_db]
    dbi_norm_max = max(dbi_inv)
    dbi_norm = [v/dbi_norm_max for v in dbi_inv]

    ch_km  = calinski_harabasz_score(X_no_pca, km_labels)
    ch_kmp = calinski_harabasz_score(X_pca,    km_pca_labels)
    ch_db  = 257.76
    ch_max = max(ch_km, ch_kmp, ch_db)

    raw_scores = {
        "K-Means (orig)": [
            silhouette_score(X_no_pca, km_labels),
            dbi_norm[0],
            ch_km / ch_max,
            max(0, adjusted_rand_score(y, km_labels)),
            normalized_mutual_info_score(y, km_labels),
        ],
        "K-Means (PCA)": [
            silhouette_score(X_pca, km_pca_labels),
            dbi_norm[1],
            ch_kmp / ch_max,
            max(0, adjusted_rand_score(y, km_pca_labels)),
            normalized_mutual_info_score(y, km_pca_labels),
        ],
        "DBSCAN (No PCA)": [
            silhouette_score(X_no_pca[db_labels != -1], db_labels[db_labels != -1]),
            dbi_norm[2],
            ch_db / ch_max,
            adjusted_rand_score(y, db_labels),
            normalized_mutual_info_score(y, db_labels),
        ],
    }

    categories = ["Silhouette", "DBI\n(inverted)", "Calinski-\nHarabasz", "ARI", "NMI"]
    N = len(categories)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7),
                            subplot_kw=dict(projection="polar"))
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    colors_radar = [PALETTE["authentic"], "#0f6e56", PALETTE["forged"]]

    for (model, scores), color in zip(raw_scores.items(), colors_radar):
        values = scores + scores[:1]
        ax.plot(angles, values, linewidth=2, linestyle="solid",
                label=model, color=color)
        ax.fill(angles, values, alpha=0.12, color=color)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=10)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(["0.2","0.4","0.6","0.8","1.0"], size=8, color="grey")
    ax.grid(color="lightgrey", linewidth=0.8)

    ax.set_title("Model Comparison — Radar Chart\n(all metrics normalised to [0, 1])",
                 size=13, fontweight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.15),
              framealpha=0.9, edgecolor="lightgrey")

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/12_radar_chart.png")
    plt.close("all")


# ─── MAIN ─────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 60)
    print("  Banknote Authentication — Visualization Suite")
    print("=" * 60)
    print("\nLoading data...")
    data = load_data()

    print("\nGenerating plots → saving to ./plots/")
    plot_boxplots(data)
    plot_outlier_scatter(data)
    plot_pca_variance(data)
    plot_pca_biplot(data)
    plot_kmeans_clusters(data)
    plot_confusion_matrices(data)
    plot_grouped_metrics(data)
    plot_radar_chart(data)

    print("\n" + "=" * 60)
    print("  Done! 8 plots saved to ./plots/")
    print("=" * 60)
    print("""
Files generated:
 
  01_boxplots_by_class.png     — Feature distributions (boxplots)
  02_outlier_flag_scatter.png  — IQR outlier flag scatter
  03_pca_explained_variance.png — PCA variance + cumulative line
  04_pca_biplot.png            — PCA biplot with loadings
  05_kmeans_cluster_plot.png   — K-Means clusters vs ground truth
  06_confusion_matrices.png    — Confusion matrix heatmaps
  07_grouped_metrics_bar.png   — All metrics grouped bar chart
  08_radar_chart.png           — Spider chart (normalised metrics)
""")

  Banknote Authentication — Visualization Suite

Loading data...
  Real data not found — generating synthetic data.

Generating plots → saving to ./plots/
  [1/8] Box plots by class...
  [2/8] Outlier flag scatter...
  [3/8] PCA explained variance...
  [4/8] PCA biplot...
  [5/8] K-Means cluster plot (PCA space)...
  [6/8] Confusion matrix heatmaps...
  [7/8] Grouped metrics bar chart...
  [8/8] Radar / spider chart...

  Done! 8 plots saved to ./plots/

Files generated:

  01_boxplots_by_class.png     — Feature distributions (boxplots)
  02_outlier_flag_scatter.png  — IQR outlier flag scatter
  03_pca_explained_variance.png — PCA variance + cumulative line
  04_pca_biplot.png            — PCA biplot with loadings
  05_kmeans_cluster_plot.png   — K-Means clusters vs ground truth
  06_confusion_matrices.png    — Confusion matrix heatmaps
  07_grouped_metrics_bar.png   — All metrics grouped bar chart
  08_radar_chart.png           — Spider chart (normalised metrics)

